In [19]:
# Cell Description: Import data-cleaning utilities, configure the project root path, and initialize the cleaning log.

import sys
import os
import json
import pandas as pd
import numpy as np

# Add the project root to sys.path to enable imports from src/
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

# Import data loading, cleaning, validation, and station-splitting utilities
from src.data import (
    load_raw_data,
    clean_column_names,
    parse_types_and_detection_limits,
    audit_and_remove_duplicates,
    filter_physically_impossible_values,
    generate_station_split
)

# Initialize a log to record data-cleaning operations
cleaning_log = []

In [20]:
# Cell Description: Load the Ireland dataset and present its dimensions, data types, sample records, summary statistics, and standardized column names in a clear, organized format.

# 1. Load the raw dataset
raw_path = "../data/raw/Ireland_dataset.csv"
df_raw = load_raw_data(raw_path)

# 2. Display dataset overview
print("=" * 80)
print("                 IRELAND WATER QUALITY DATASET — OVERVIEW")
print("=" * 80)

print(f"\n📊 Dataset Dimensions : {df_raw.shape[0]:,} rows × {df_raw.shape[1]} columns")
print(f"💾 Memory Usage       : {df_raw.memory_usage(deep=True).sum() / (1024 ** 2):.2f} MB")
print(f"🔢 Numerical Columns  : {df_raw.select_dtypes(include=np.number).shape[1]}")
print(f"🔤 Categorical Columns: {df_raw.select_dtypes(include=['object', 'category']).shape[1]}")

# 3. Preview sample records
print("\n" + "=" * 80)
print("                         SAMPLE RECORDS")
print("=" * 80)
display(df_raw.head(5).style.set_caption("First 5 Records").set_properties(**{
    "text-align": "left",
    "max-width": "220px",
    "white-space": "normal"
}))

# 4. Inspect column data types and missing values
print("\n" + "=" * 80)
print("                    COLUMN INFORMATION")
print("=" * 80)

column_info = pd.DataFrame({
    "Data Type": df_raw.dtypes.astype(str),
    "Non-Null Count": df_raw.notna().sum(),
    "Missing Count": df_raw.isna().sum(),
    "Missing (%)": (df_raw.isna().mean() * 100).round(2),
    "Unique Values": df_raw.nunique()
})

display(
    column_info.style
    .format({"Missing (%)": "{:.2f}%"})
    .background_gradient(subset=["Missing (%)"], cmap="YlOrRd")
    .set_caption("Column Quality Report")
)

# 5. Display numerical summary statistics
print("\n" + "=" * 80)
print("                    NUMERICAL SUMMARY STATISTICS")
print("=" * 80)

numeric_summary = df_raw.describe().T.round(2)
display(numeric_summary.style.set_caption("Descriptive Statistics"))

# 6. Standardize column names
df = clean_column_names(df_raw)

print("\n" + "=" * 80)
print("                    STANDARDIZED COLUMN NAMES")
print("=" * 80)

column_names = pd.DataFrame({
    "Column Number": range(1, len(df.columns) + 1),
    "Standardized Column Name": df.columns
})

display(column_names.style.hide(axis="index").set_caption("Clean Column Headers"))

print("\n✅ Dataset loaded and column names standardized successfully.")

                 IRELAND WATER QUALITY DATASET — OVERVIEW

📊 Dataset Dimensions : 235,019 rows × 14 columns
💾 Memory Usage       : 84.59 MB
🔢 Numerical Columns  : 9
🔤 Categorical Columns: 5

                         SAMPLE RECORDS


,Country,Area,Waterbody Type,Date,Ammonia (mg/l),Biochemical Oxygen Demand (mg/l),Dissolved Oxygen (mg/l),Orthophosphate (mg/l),pH (ph units),Temperature (cel),Nitrogen (mg/l),Nitrate (mg/l),CCME_Values,CCME_WQI
0,Ireland,"Ballyteigue-Bannow, Adamstown",River,07-07-2010,0.038000,1.400000,5.940000,0.025000,6.000000,11.500000,3.110000,5.200000,100.000000,Excellent
1,Ireland,"Ballyteigue-Bannow, Adamstown",River,14-09-2010,0.038000,1.400000,11.684103,0.025000,6.500000,11.100000,3.390000,2.400000,100.000000,Excellent
2,Ireland,"Ballyteigue-Bannow, Adamstown",River,17-11-2010,0.038000,1.400000,5.850000,0.025000,6.300000,11.000000,3.930000,4.500000,100.000000,Excellent
3,Ireland,"Ballyteigue-Bannow, Adamstown",River,09-03-2011,0.038000,1.400000,6.030000,0.025000,6.200000,10.900000,3.200000,4.200000,100.000000,Excellent
4,Ireland,"Ballyteigue-Bannow, Adamstown",River,29-06-2011,0.038000,1.400000,3.420000,0.025000,6.200000,11.100000,2.700000,4.300000,100.000000,Excellent



                    COLUMN INFORMATION


,Data Type,Non-Null Count,Missing Count,Missing (%),Unique Values
Country,object,235019,0,0.00%,1
Area,object,235019,0,0.00%,9737
Waterbody Type,object,235019,0,0.00%,4
Date,object,235019,0,0.00%,4276
Ammonia (mg/l),float64,235019,0,0.00%,5879
Biochemical Oxygen Demand (mg/l),float64,235019,0,0.00%,1628
Dissolved Oxygen (mg/l),float64,235019,0,0.00%,13563
Orthophosphate (mg/l),float64,235019,0,0.00%,5001
pH (ph units),float64,235019,0,0.00%,3160
Temperature (cel),float64,235019,0,0.00%,4224



                    NUMERICAL SUMMARY STATISTICS


,count,mean,std,min,25%,50%,75%,max
Ammonia (mg/l),235019.000000,0.080000,0.940000,-0.000000,0.030000,0.040000,0.040000,134.500000
Biochemical Oxygen Demand (mg/l),235019.000000,1.530000,1.090000,-2.000000,1.400000,1.400000,1.400000,220.000000
Dissolved Oxygen (mg/l),235019.000000,8.510000,3.230000,0.100000,4.970000,9.090000,11.680000,20.000000
Orthophosphate (mg/l),235019.000000,0.220000,2.590000,-0.000000,0.020000,0.020000,0.030000,100.000000
pH (ph units),235019.000000,7.680000,0.850000,3.200000,7.400000,7.800000,8.060000,30.000000
Temperature (cel),235019.000000,11.320000,4.010000,0.000000,8.500000,11.150000,14.000000,98.000000
Nitrogen (mg/l),235019.000000,1.780000,1.510000,0.000000,0.920000,1.420000,2.100000,46.000000
Nitrate (mg/l),235019.000000,1.510000,1.170000,0.000000,1.300000,1.300000,1.300000,112.450000
CCME_Values,235019.000000,98.110000,6.410000,37.190000,100.000000,100.000000,100.000000,100.000000



                    STANDARDIZED COLUMN NAMES


Column Number,Standardized Column Name
1,country
2,site_id
3,waterbody_type
4,date
5,ammonia_mg_l
6,bod_mg_l
7,dissolved_oxygen_mg_l
8,orthophosphate_mg_l
9,ph
10,temperature_c



✅ Dataset loaded and column names standardized successfully.


In [21]:
# Cell Description: Convert water-quality measurements to numeric types, handle detection limits, log unit consistency checks, and verify date conversion.

# 1. Define water-quality measurement columns
numeric_measurement_cols = [
    'ammonia_mg_l',
    'bod_mg_l',
    'dissolved_oxygen_mg_l',
    'orthophosphate_mg_l',
    'ph',
    'temperature_c',
    'nitrogen_mg_l',
    'nitrate_mg_l'
]

# 2. Parse numeric values and detection limits
df, type_logs = parse_types_and_detection_limits(
    df,
    numeric_measurement_cols
)
cleaning_log.extend(type_logs)

# 3. Record the unit consistency audit
cleaning_log.append({
    'Issue found': 'Unit consistency audit',
    'How many rows/values': df.shape[0],
    'Action': 'Verified measurement columns for consistent units',
    'Why': 'Column headers indicate mg/L, °C, and standard pH units; homogeneous units assumed based on available metadata'
})

# 4. Display measurement conversion summary
print("=" * 75)
print("        WATER QUALITY MEASUREMENT & DATE VALIDATION")
print("=" * 75)

print(f"\nDataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Measurement Columns Checked: {len(numeric_measurement_cols)}")

print("\n--- Measurement Data Types ---")
measurement_info = pd.DataFrame({
    "Data Type": df[numeric_measurement_cols].dtypes.astype(str),
    "Missing Values": df[numeric_measurement_cols].isna().sum(),
    "Missing (%)": (df[numeric_measurement_cols].isna().mean() * 100).round(2)
})
display(measurement_info.style.format({"Missing (%)": "{:.2f}%"}))

# 5. Verify date conversion
print("\n--- Date Conversion Status ---")

if 'date' in df.columns:
    print(f"Date Data Type : {df['date'].dtype}")
    print(f"Earliest Date  : {df['date'].min()}")
    print(f"Latest Date    : {df['date'].max()}")
    print(f"Missing Dates  : {df['date'].isna().sum():,}")
else:
    print("⚠️ Date column not found. Check the standardized column names.")

print("\n✅ Measurement parsing and validation checks completed.")

        WATER QUALITY MEASUREMENT & DATE VALIDATION

Dataset Shape: 235,019 rows × 14 columns
Measurement Columns Checked: 8

--- Measurement Data Types ---


,Data Type,Missing Values,Missing (%)
ammonia_mg_l,float64,0,0.00%
bod_mg_l,float64,0,0.00%
dissolved_oxygen_mg_l,float64,0,0.00%
orthophosphate_mg_l,float64,0,0.00%
ph,float64,0,0.00%
temperature_c,float64,0,0.00%
nitrogen_mg_l,float64,0,0.00%
nitrate_mg_l,float64,0,0.00%



--- Date Conversion Status ---
Date Data Type : datetime64[ns]
Earliest Date  : 1940-11-10 00:00:00
Latest Date    : 2023-12-17 00:00:00
Missing Dates  : 0

✅ Measurement parsing and validation checks completed.


In [22]:
# Cell Description: Detect and remove duplicate records, log the cleaning actions, and display the updated dataset size.

# 1. Audit and remove duplicate records
df, dup_logs = audit_and_remove_duplicates(df)

# 2. Record duplicate-removal details
cleaning_log.extend(dup_logs)

# 3. Display the post-cleaning summary
print("=" * 65)
print("             DUPLICATE RECORD AUDIT")
print("=" * 65)

print(f"Remaining Rows   : {len(df):,}")
print(f"Remaining Columns: {df.shape[1]:,}")
print(f"Duplicate Rows   : {df.duplicated().sum():,}")

print("\n✅ Duplicate audit completed and cleaning log updated.")

             DUPLICATE RECORD AUDIT
Remaining Rows   : 234,993
Remaining Columns: 14
Duplicate Rows   : 0

✅ Duplicate audit completed and cleaning log updated.


In [23]:
# Cell Description: Apply environmental chemistry constraints to remove physically impossible measurements while preserving genuine pollution-related outliers.

# 1. Define physical domain constraints for water-quality parameters
domain_rules = {
    'ph': (
        0.0, 14.0,
        'Outside physical pH scale; possible sensor or measurement error'
    ),
    'ammonia_mg_l': (
        0.0, 100.0,
        'Negative concentration is impossible; extreme values require validation'
    ),
    'bod_mg_l': (
        0.0, 150.0,
        'Negative concentration is impossible; high values may indicate severe pollution'
    ),
    'dissolved_oxygen_mg_l': (
        0.0, 25.0,
        'Negative concentration is impossible; unusually high values require validation'
    ),
    'orthophosphate_mg_l': (
        0.0, 50.0,
        'Negative phosphorus concentration is physically impossible'
    ),
    'temperature_c': (
        -5.0, 45.0,
        'Values outside expected ambient water temperatures require validation'
    ),
    'nitrogen_mg_l': (
        0.0, 200.0,
        'Negative nitrogen concentration is physically impossible'
    ),
    'nitrate_mg_l': (
        0.0, 200.0,
        'Negative nitrate concentration is physically impossible'
    )
}

# 2. Filter measurements outside the defined physical limits
df, domain_logs = filter_physically_impossible_values(
    df,
    domain_rules
)

# 3. Record domain-validation results in the cleaning log
cleaning_log.extend(domain_logs)

# 4. Document the outlier-handling strategy
cleaning_log.append({
    'Issue found': 'Distribution outlier evaluation (extreme pollution events)',
    'How many rows/values': 'Monitored across parameters',
    'Action': 'Preserved statistical outliers within the defined physical limits',
    'Why': 'Genuine pollution events may produce extreme measurements important for water-quality analysis and model training'
})

# 5. Display the validation summary
print("=" * 75)
print("          ENVIRONMENTAL DOMAIN VALIDATION SUMMARY")
print("=" * 75)

print(f"\nRemaining Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Parameters Validated: {len(domain_rules)}")
print(f"Cleaning Log Entries: {len(cleaning_log)}")

print("\n--- Applied Physical Limits ---")
domain_summary = pd.DataFrame([
    {
        "Parameter": col,
        "Minimum": limits[0],
        "Maximum": limits[1],
        "Validation Rule": limits[2]
    }
    for col, limits in domain_rules.items()
])

display(domain_summary)

print("\n✅ Domain validation completed; retained in-range statistical outliers.")

          ENVIRONMENTAL DOMAIN VALIDATION SUMMARY

Remaining Dataset Shape: 234,993 rows × 14 columns
Parameters Validated: 8
Cleaning Log Entries: 11

--- Applied Physical Limits ---


,Parameter,Minimum,Maximum,Validation Rule
0,ph,0.0,14.0,Outside physical pH scale; possible sensor or ...
1,ammonia_mg_l,0.0,100.0,Negative concentration is impossible; extreme ...
2,bod_mg_l,0.0,150.0,Negative concentration is impossible; high val...
3,dissolved_oxygen_mg_l,0.0,25.0,Negative concentration is impossible; unusuall...
4,orthophosphate_mg_l,0.0,50.0,Negative phosphorus concentration is physicall...
5,temperature_c,-5.0,45.0,Values outside expected ambient water temperat...
6,nitrogen_mg_l,0.0,200.0,Negative nitrogen concentration is physically ...
7,nitrate_mg_l,0.0,200.0,Negative nitrate concentration is physically i...



✅ Domain validation completed; retained in-range statistical outliers.


In [24]:
# Cell Description: Remove the constant country column, inspect missing values after domain filtering, and verify target integrity.

# 1. Remove the country column if it contains a single unique non-null value
if 'country' in df.columns and df['country'].nunique(dropna=False) == 1:
    df = df.drop(columns=['country'])

    cleaning_log.append({
        'Issue found': 'Uninformative constant column (country)',
        'How many rows/values': len(df),
        'Action': 'Dropped column country',
        'Why': 'A constant column provides no predictive information'
    })

# 2. Display missing-value percentages after domain filtering
print("=" * 70)
print("       MISSING DATA & TARGET INTEGRITY REPORT")
print("=" * 70)

missing_post = (df.isnull().mean() * 100).round(3)

print("\n--- Missing Values by Column (%) ---")
display(
    missing_post.rename("Missing (%)")
    .to_frame()
    .style.format("{:.3f}%")
    .background_gradient(cmap="YlOrRd")
)

# 3. Verify that both target columns contain no missing values
assert df['target_ccme_value'].isnull().sum() == 0, \
    "Target CCME_Values contains missing values!"

assert df['target_ccme_class'].isnull().sum() == 0, \
    "Target CCME_WQI contains missing values!"

# 4. Display final validation summary
print("\n--- Target Integrity Check ---")
print("Continuous Target (target_ccme_value): 0 missing values")
print("Categorical Target (target_ccme_class): 0 missing values")
print(f"\nFinal Dataset Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Cleaning Log Entries: {len(cleaning_log)}")

print("\n✅ Target integrity confirmed.")

       MISSING DATA & TARGET INTEGRITY REPORT

--- Missing Values by Column (%) ---


,Missing (%)
site_id,0.000%
waterbody_type,0.000%
date,0.000%
ammonia_mg_l,0.004%
bod_mg_l,0.001%
dissolved_oxygen_mg_l,0.000%
orthophosphate_mg_l,0.064%
ph,0.213%
temperature_c,0.052%
nitrogen_mg_l,0.000%



--- Target Integrity Check ---
Continuous Target (target_ccme_value): 0 missing values
Categorical Target (target_ccme_class): 0 missing values

Final Dataset Shape: 234,993 rows × 13 columns
Cleaning Log Entries: 12

✅ Target integrity confirmed.


In [25]:
# Cell Description: Display the data-cleaning audit log and export a Markdown report documenting dataset changes and cleaning decisions.

# 1. Convert the cleaning log into a structured DataFrame
log_df = pd.DataFrame(cleaning_log)

# 2. Display the cleaning log in a readable table
print("=" * 80)
print("                    DATA CLEANING AUDIT LOG")
print("=" * 80)

if not log_df.empty:
    display(
        log_df.style
        .set_caption("Ireland Water Quality — Cleaning Report")
        .set_properties(**{
            "text-align": "left",
            "white-space": "normal",
            "max-width": "300px"
        })
    )
else:
    print("No cleaning operations have been logged.")

# 3. Define the Markdown report path
markdown_log_path = "../docs/cleaning_log.md"

# 4. Create the output directory if it does not exist
os.makedirs(os.path.dirname(markdown_log_path), exist_ok=True)

# 5. Export the cleaning log with dataset information
with open(markdown_log_path, "w", encoding="utf-8") as f:
    f.write("# Data Cleaning Log (Section 5.3)\n\n")
    f.write("**Dataset:** Ireland Regional Surface Water Quality\n\n")
    f.write(f"**Input Rows:** {len(df_raw):,}  \n")
    f.write(f"**Output Clean Rows:** {len(df):,}  \n")
    f.write(f"**Rows Removed:** {len(df_raw) - len(df):,}  \n")
    f.write(f"**Final Columns:** {df.shape[1]:,}  \n")
    f.write(f"**Cleaning Operations Logged:** {len(log_df):,}\n\n")

    if not log_df.empty:
        f.write(log_df.to_markdown(index=False))
        f.write("\n")

# 6. Confirm report export
print("\n" + "=" * 80)
print("                    REPORT EXPORT STATUS")
print("=" * 80)
print(f"Report Path: {os.path.abspath(markdown_log_path)}")
print(f"Log Entries: {len(log_df):,}")
print("✅ Cleaning log report saved successfully.")

                    DATA CLEANING AUDIT LOG


,Issue found,How many rows/values,Action,Why
0,Unit consistency audit,235019,Verified measurement columns for consistent units,"Column headers indicate mg/L, °C, and standard pH units; homogeneous units assumed based on available metadata"
1,Exact duplicate rows across all columns,26,Dropped duplicate rows,Redundant sampling records cause artificial weighting and data leakage
2,"ph outside physical domain [0.0, 14.0]",501,Set to NaN (preserved row),Outside physical pH scale; possible sensor or measurement error
3,"ammonia_mg_l outside physical domain [0.0, 100.0]",9,Set to NaN (preserved row),Negative concentration is impossible; extreme values require validation
4,"bod_mg_l outside physical domain [0.0, 150.0]",3,Set to NaN (preserved row),Negative concentration is impossible; high values may indicate severe pollution
5,"dissolved_oxygen_mg_l physical bounds check [0.0, 25.0]",0,None (all values in valid physical range),Negative concentration is impossible; unusually high values require validation
6,"orthophosphate_mg_l outside physical domain [0.0, 50.0]",150,Set to NaN (preserved row),Negative phosphorus concentration is physically impossible
7,"temperature_c outside physical domain [-5.0, 45.0]",123,Set to NaN (preserved row),Values outside expected ambient water temperatures require validation
8,"nitrogen_mg_l physical bounds check [0.0, 200.0]",0,None (all values in valid physical range),Negative nitrogen concentration is physically impossible
9,"nitrate_mg_l physical bounds check [0.0, 200.0]",0,None (all values in valid physical range),Negative nitrate concentration is physically impossible



                    REPORT EXPORT STATUS
Report Path: /Users/sarthakmathapati/Desktop/AML:dl/aata/docs/cleaning_log.md
Log Entries: 12
✅ Cleaning log report saved successfully.


In [26]:
# Cell Description: Export the cleaned Ireland water-quality dataset to CSV and display the saved file path and final dataset dimensions.

# 1. Define the output file path
processed_csv_path = "../data/processed/clean.csv"

# 2. Create the output directory if it does not exist
os.makedirs(os.path.dirname(processed_csv_path), exist_ok=True)

# 3. Save the cleaned dataset
df.to_csv(processed_csv_path, index=False, encoding="utf-8")

# 4. Display the export summary
print("=" * 70)
print("           CLEANED DATASET EXPORT SUMMARY")
print("=" * 70)

print(f"Output File : {os.path.abspath(processed_csv_path)}")
print(f"Total Rows  : {df.shape[0]:,}")
print(f"Total Columns: {df.shape[1]:,}")
print(f"File Size   : {os.path.getsize(processed_csv_path) / (1024 ** 2):.2f} MB")

print("\n✅ Cleaned dataset saved successfully.")

           CLEANED DATASET EXPORT SUMMARY
Output File : /Users/sarthakmathapati/Desktop/AML:dl/aata/data/processed/clean.csv
Total Rows  : 234,993
Total Columns: 13
File Size   : 25.76 MB

✅ Cleaned dataset saved successfully.


In [27]:
# Cell Description: Split data by monitoring station, verify train-test isolation to prevent data leakage, and save split indices and metadata.

# 1. Generate station-based train-test indices
train_idx, test_idx = generate_station_split(
    df,
    group_col="site_id",
    test_size=0.20,
    seed=42
)

# 2. Identify stations in each split
train_stations = set(df.iloc[train_idx]["site_id"].unique())
test_stations = set(df.iloc[test_idx]["site_id"].unique())

# 3. Verify that no stations appear in both splits
station_overlap = train_stations.intersection(test_stations)

# 4. Display split verification report
print("=" * 70)
print("             TRAIN-TEST SPLIT VERIFICATION")
print("=" * 70)

print(f"\nTotal Samples       : {len(df):,}")
print(f"Training Samples   : {len(train_idx):,} ({len(train_idx) / len(df) * 100:.1f}%)")
print(f"Testing Samples    : {len(test_idx):,} ({len(test_idx) / len(df) * 100:.1f}%)")
print(f"Training Stations  : {len(train_stations):,}")
print(f"Testing Stations   : {len(test_stations):,}")
print(f"Overlapping Stations: {len(station_overlap):,}")

# Stop execution if station leakage is detected
assert len(station_overlap) == 0, (
    f"Data leakage detected: {len(station_overlap)} stations overlap "
    "between training and testing sets!"
)

print("\n✅ Station isolation verified: no train-test station overlap.")

# 5. Prepare split metadata and indices for export
split_data = {
    "split_type": "GroupShuffleSplit",
    "group_column": "site_id",
    "test_size": 0.20,
    "random_state": 42,
    "train_samples": len(train_idx),
    "test_samples": len(test_idx),
    "train_idx": [int(i) for i in train_idx],
    "test_idx": [int(i) for i in test_idx]
}

# 6. Save split indices and metadata
split_json_path = "../data/processed/split_indices.json"

os.makedirs(os.path.dirname(split_json_path), exist_ok=True)

with open(split_json_path, "w", encoding="utf-8") as f:
    json.dump(split_data, f, indent=4)

# 7. Confirm successful export
print("\n" + "=" * 70)
print("                 SPLIT EXPORT SUMMARY")
print("=" * 70)

print(f"Output File : {os.path.abspath(split_json_path)}")
print(f"Train Size  : {len(train_idx):,} samples")
print(f"Test Size   : {len(test_idx):,} samples")
print("Status      : Successfully saved")

print("\n✅ Split indices and metadata exported successfully.")

             TRAIN-TEST SPLIT VERIFICATION

Total Samples       : 234,993
Training Samples   : 190,216 (80.9%)
Testing Samples    : 44,777 (19.1%)
Training Stations  : 7,789
Testing Stations   : 1,948
Overlapping Stations: 0

✅ Station isolation verified: no train-test station overlap.

                 SPLIT EXPORT SUMMARY
Output File : /Users/sarthakmathapati/Desktop/AML:dl/aata/data/processed/split_indices.json
Train Size  : 190,216 samples
Test Size   : 44,777 samples
Status      : Successfully saved

✅ Split indices and metadata exported successfully.
